# 06 - Pipeline de ingestión para reentrenamiento

Este notebook documenta y valida el sistema de recogida de datos nuevos implementado en la aplicación Streamlit.

La funcionalidad principal del pipeline está integrada en `app/app.py`, porque los datos nuevos se generan cuando un usuario utiliza la app. Este notebook sirve como apoyo técnico para explicar el flujo, revisar los archivos generados y construir el dataset final de reentrenamiento de forma reproducible.

## Objetivo

Convertir registros generados por la app en un dataset supervisado apto para futuros reentrenamientos.

Para reentrenar un modelo de regresión no basta con guardar la predicción. Hace falta conocer el valor real observado de `FloodProbability`. Por eso el pipeline diferencia entre registros pendientes y registros validados.

## Flujo del pipeline

1. El usuario introduce variables en la vista **Predicción** de Streamlit.
2. La app guarda cada predicción en `data/new_data/nuevos_registros.csv`.
3. Si todavía no se conoce el valor real, el registro queda como `pending_target`.
4. Cuando se añade el valor real desde **Monitorización**, el registro pasa a `validated_for_retraining`.
5. La vista **Pipeline de reentrenamiento** genera `data/processed/retraining_dataset.csv` usando solo registros validados.

La predicción del modelo se conserva como referencia, pero no se usa como variable objetivo para reentrenar. La variable objetivo del dataset de reentrenamiento es el valor real observado.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NEW_DATA_PATH = PROJECT_ROOT / "data" / "new_data" / "nuevos_registros.csv"
RETRAINING_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "retraining_dataset.csv"

FEATURE_COLUMNS = [
    "MonsoonIntensity",
    "TopographyDrainage",
    "RiverManagement",
    "Deforestation",
    "Urbanization",
    "ClimateChange",
    "DamsQuality",
    "Siltation",
    "AgriculturalPractices",
    "Encroachments",
    "IneffectiveDisasterPreparedness",
    "DrainageSystems",
    "CoastalVulnerability",
    "Landslides",
    "Watersheds",
    "DeterioratingInfrastructure",
    "PopulationScore",
    "WetlandLoss",
    "InadequatePlanning",
    "PoliticalFactors",
]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"NEW_DATA_PATH existe: {NEW_DATA_PATH.exists()} -> {NEW_DATA_PATH}")
print(f"RETRAINING_DATASET_PATH: {RETRAINING_DATASET_PATH}")

PROJECT_ROOT: C:\Users\migue\Documents\Proyecto Regresión
NEW_DATA_PATH existe: True -> C:\Users\migue\Documents\Proyecto Regresión\data\new_data\nuevos_registros.csv
RETRAINING_DATASET_PATH: C:\Users\migue\Documents\Proyecto Regresión\data\processed\retraining_dataset.csv


## Carga de registros nuevos

El archivo `nuevos_registros.csv` es local y está ignorado por Git. Por eso este notebook puede ejecutarse en dos situaciones:

- Si existe el CSV, se revisan los registros reales generados por la app.
- Si no existe, se muestra una estructura vacía y se explica cómo generarlo desde Streamlit.

In [2]:
def load_new_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        print("No existe nuevos_registros.csv. Genera predicciones desde Streamlit para alimentar el pipeline.")
        return pd.DataFrame()

    df = pd.read_csv(path)
    if "actual_value" not in df.columns:
        df["actual_value"] = pd.NA
    if "record_status" not in df.columns:
        df["record_status"] = pd.NA

    df["actual_value"] = pd.to_numeric(df["actual_value"], errors="coerce")
    df["record_status"] = df["actual_value"].apply(
        lambda value: "validated_for_retraining" if pd.notna(value) else "pending_target"
    )
    return df

new_data_df = load_new_data(NEW_DATA_PATH)
print(f"Registros cargados: {len(new_data_df)}")
new_data_df.tail(5)

Registros cargados: 2


,timestamp,consultor,model_name,model_version,model_file,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,...,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,prediction,prediction_id,actual_value,record_status
0,2026-06-23T12:44:27,Estudiante,Linear Regression Baseline,baseline_v1,flood_baseline_model.joblib,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,0.511411,new_legacy_0_20260623T124427,NaN,pending_target
1,2026-06-23T12:45:17,Estudiante,Linear Regression Baseline,baseline_v1,flood_baseline_model.joblib,12.0,17.0,14.0,2.0,2.0,...,16.0,14.0,17.0,18.0,15.0,2.0,1.000000,new_legacy_1_20260623T124517,NaN,pending_target


## Estado de los registros

El pipeline separa los registros en dos grupos:

- `pending_target`: registros con variables y predicción, pero sin valor real observado.
- `validated_for_retraining`: registros con valor real observado, aptos para reentrenamiento.

Solo el segundo grupo puede formar parte del dataset supervisado.

In [3]:
if new_data_df.empty:
    status_summary = pd.DataFrame(columns=["record_status", "num_registros"])
else:
    status_summary = (
        new_data_df["record_status"]
        .fillna("pending_target")
        .value_counts()
        .rename_axis("record_status")
        .reset_index(name="num_registros")
    )

status_summary

,record_status,num_registros
0,pending_target,2


## Construcción del dataset de reentrenamiento

El dataset final debe tener:

- Las 20 variables predictoras usadas por el modelo.
- La variable objetivo `FloodProbability`, tomada desde `actual_value`.

No se usa `prediction` como target, porque eso solo repetiría lo que dijo el modelo anterior. Para reentrenar necesitamos el valor real observado.

In [4]:
def build_retraining_dataset(new_data: pd.DataFrame) -> pd.DataFrame:
    if new_data.empty:
        return pd.DataFrame(columns=[*FEATURE_COLUMNS, "FloodProbability"])

    required_cols = [*FEATURE_COLUMNS, "actual_value"]
    missing_cols = [col for col in required_cols if col not in new_data.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas necesarias para reentrenar: {missing_cols}")

    validated_df = new_data.dropna(subset=["actual_value"]).copy()
    retraining_df = validated_df[FEATURE_COLUMNS].copy()
    retraining_df["FloodProbability"] = validated_df["actual_value"].astype(float)
    return retraining_df

retraining_df = build_retraining_dataset(new_data_df)
print(f"Registros listos para reentrenamiento: {len(retraining_df)}")
retraining_df.tail(5)

Registros listos para reentrenamiento: 0


,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability


## Guardado del dataset validado

La aplicación Streamlit permite generar este mismo archivo desde la vista **Pipeline de reentrenamiento**.

Este notebook reproduce esa lógica y guarda el resultado en `data/processed/retraining_dataset.csv` cuando existen registros validados.

In [5]:
if retraining_df.empty:
    print("No se genera retraining_dataset.csv porque no hay registros con valor real observado.")
else:
    RETRAINING_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    retraining_df.to_csv(RETRAINING_DATASET_PATH, index=False)
    print(f"Dataset de reentrenamiento generado en: {RETRAINING_DATASET_PATH}")

No se genera retraining_dataset.csv porque no hay registros con valor real observado.


## Validaciones mínimas

Antes de usar el dataset para reentrenar, se comprueba que:

- Tiene la columna objetivo `FloodProbability`.
- Mantiene las 20 variables predictoras esperadas.
- No incluye registros sin valor real observado.
- La variable objetivo está en rango 0-1.

In [6]:
if retraining_df.empty:
    print("Sin registros validados: validaciones no aplicables todavía.")
else:
    assert "FloodProbability" in retraining_df.columns
    assert all(col in retraining_df.columns for col in FEATURE_COLUMNS)
    assert retraining_df["FloodProbability"].notna().all()
    assert retraining_df["FloodProbability"].between(0, 1).all()
    print("Validaciones mínimas superadas.")

Sin registros validados: validaciones no aplicables todavía.


## Conclusión

El pipeline de ingestión queda integrado principalmente en Streamlit, porque es la app la que recoge los datos durante el uso real.

Este notebook documenta y reproduce la lógica principal:

- Leer registros nuevos generados por la app.
- Identificar registros pendientes y registros validados.
- Construir un dataset supervisado usando `actual_value` como `FloodProbability`.
- Guardar el dataset de reentrenamiento en `data/processed/retraining_dataset.csv`.

Con esto se cubre el requisito del Nivel Medio relacionado con la recogida de datos nuevos para futuros reentrenamientos.